In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.stats import bernoulli
import numpy as np
import math
import scipy.stats
import random 
import scipy
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
def drawGraph (G, pos = None):
    plt.figure (figsize=(15,15))
    if not pos:
        pos = nx.spring_layout(G)
    nx.draw(G, pos)
    nx.draw_networkx_labels(G, pos) 
    plt.show()


In [ ]:
# 2.a
G = nx.DiGraph ()
G.add_edge (0,1)
drawGraph (G)
print (nx.adjacency_matrix (G).todense())

In [ ]:
#2d 
def graphFromEdgesFile (fname):
    G = nx.read_edgelist (fname, delimiter = '\t')
    print (len (G.nodes ()), len (G.edges ()))
    return G

# http://konect.uni-koblenz.de/networks/dolphins
Gd = graphFromEdgesFile ('out.dolphins')

# http://konect.uni-koblenz.de/networks/arenas-jazz
Gj = graphFromEdgesFile ('out.arenas-jazz')

drawGraph (Gd)
drawGraph (Gj)

In [ ]:
# 2f

plt.hist ([f[1] for f in Gd.degree()])
plt.show ()

plt.hist ([f[1] for f in Gj.degree()])
plt.show ()

In [ ]:
# 3
G = nx.Graph ()
G.add_edge (0,1)
G.add_edge (0,2)
G.add_edge (2,1)
G.add_edge (2,3)
drawGraph (G)

print ('local clustering:', nx.clustering (G))
print ('mean local clustering:', nx.average_clustering (G))
print ('global clustering = 3 / 5 = ', 3.0 / 5 )

print (nx.adjacency_matrix(G).todense ())

In [ ]:
# ex. IV.4
import random


def genPair (nodesNumber):
    fr = random.randint (0, nodesNumber - 1)
    to = random.randint (0, nodesNumber - 1)
    if fr == to:
        return genPair (nodesNumber)
    return fr,to
    
def modelC (nodesNumber, k, shortcutsNumber, withoutDuplicates):
    G = nx.Graph ()
    for i in range (nodesNumber):
        for j in range (i + 1, i + k + 1):
            G.add_edge (i, j % nodesNumber)
    for i in range (shortcutsNumber):
        fr,to = genPair (nodesNumber)
        while withoutDuplicates and G.has_edge (fr, to):
            fr,to = genPair (nodesNumber)
        G.add_edge (fr, to)
            
    return G

N = 35
k = 3

G = modelC (N, k, 7, True)
drawGraph (G, pos = nx.circular_layout (G))


cc = []
diam = []
xs = []
for i in range (0, 300, 10):
    G = modelC (N, k, i, False)
    xs.append (i)
    cc.append (nx.average_clustering(G))
    diam.append (nx.diameter (G))
    
plt.plot (xs, cc, label='C.coeff')
plt.plot (xs, diam, label='diameter')
plt.legend ()
plt.show ()
               
    

In [ ]:
# http://konect.uni-koblenz.de/networks/arenas-jazz
Gj = graphFromEdgesFile ('out.arenas-jazz')

clC = nx.closeness_centrality (Gj)
bC = nx.betweenness_centrality (Gj)
kC = nx.katz_centrality_numpy (Gj)
prC = nx.pagerank (Gj)
degC = nx.degree_centrality (Gj)

keys = Gj.nodes ()
correlations = [[clC[k] for k in keys], [bC[k] for k in keys], [kC[k] for k in keys], [prC[k] for k in keys], [degC[k] for k in keys]]
names = ['Closeness', 'Betweenness', 'Katz', 'Page rank', 'Degree']


plt.figure (figsize=(15,10))
for i in range (5):
    plt.plot (correlations[i], label = names[i])
plt.legend ()
plt.show ()

from scipy import stats


for i in range (5):
    for j in range (i + 1, 5):
        print (names[i], ' vs ', names[j], ': PearsonResult', stats.pearsonr (correlations[i], correlations[j]), stats.spearmanr (correlations[i], correlations[j]))

In [ ]:
#ex 2

def findGCSandMeanC (n, p):
    G = nx.erdos_renyi_graph (n, p)
    return np.mean (list(dict(nx.degree(G)).values())), 1.0 * (len ( sorted(nx.connected_components(G), key=len, reverse=True) [0]) -1 ) / n

n = 100
repeatNum = 10
vals = []

for p in np.linspace (0, 0.1, 100):
    C = 0.0
    S = 0.0
    for r in range (repeatNum):
        c,s = findGCSandMeanC(n, p)
        C += c
        S += s
    C /= repeatNum
    S /= repeatNum
    #if C < 4:
    vals.append ((C,S))
    #print(p,C,S)
vals.sort ()
plt.plot ([v[0] for v in vals], [v[1] for v in vals])
plt.xlabel('Mean degree C')
plt.ylabel('Size of giant component S')

plt.show ()

repeatNum = 10

vals = []

for p in np.linspace (0, 0.4, 50):
    MLC = 0.0
    for r in range (repeatNum):
        G = nx.erdos_renyi_graph (n, p)
        MLC += nx.average_clustering (G) / repeatNum
    vals.append ((p,MLC))
plt.plot ([v[0] for v in vals], [v[1] for v in vals])
plt.xlabel('p')
plt.ylabel('Mean local clustering')
plt.show ()


In [ ]:
#ex 3

from scipy.stats.stats import pearsonr   
import random

def pageRankVsDegree (G):
    degs = nx.degree (G)
    pagerank = nx.pagerank (G)
    fixed_order_degs = []
    fixed_order_pagerank = []
    for node in G.nodes ():
        fixed_order_degs.append (degs[node])
        fixed_order_pagerank.append (pagerank[node])
    return pearsonr(fixed_order_degs, fixed_order_pagerank)

def genRandomNet (degrees):
    G = nx.Graph ()
    n = len (degrees)
    s = sum (degrees)
    if s % 2 == 1:
        print ('odd sum...')
        return 
    sl = 0
    me = 0
    while s > 0:
        v = random.randint (0, s - 1)
        v1 = 0
        while degrees[v1] < v:
            v -= degrees[v1]
            v1 += 1
        s -= 1
        degrees[v1] -= 1
        v = random.randint (0, s - 1)
        v2 = 0
        while degrees[v2] < v:
            v -= degrees[v2]
            v2 += 1
        s -= 1
        degrees[v2] -= 1
        
        if v1 == v2:
            sl += 1
        elif (v1 in G) and (v2 in G[v1]):
            me += 1
        else:
            G.add_edge (v1, v2)
    return G, sl, me

originalG = nx.karate_club_graph ()
G, sl, me = genRandomNet (list(dict(nx.degree (originalG)).values ()))
print ('size: ', len (G.nodes ()), 'self loups: ', sl, ' multiple edges: ', me, ' sum of degrees: ', sum (dict(nx.degree(G)).values()) + 2 * sl + 2 * me, 'pagerank - degree correlation: ', pageRankVsDegree (G), ' original graph correlation: ', pageRankVsDegree (originalG)) 
nx.draw (G)
plt.show ()

plt.figure (figsize=(15,10))
originalG = nx.erdos_renyi_graph (300, 0.01)
G, sl, me = genRandomNet (list(dict(nx.degree (originalG)).values ()))
print ('size: ', len (G.nodes ()), 'self loups: ', sl, ' multiple edges: ', me, ' sum of degrees: ', sum (dict(nx.degree(G)).values()) + 2 * sl + 2 * me, 'pagerank - degree correlation: ', pageRankVsDegree (G), ' original graph correlation: ', pageRankVsDegree (originalG))
nx.draw (G)
plt.show ()
        
        

In [ ]:
#ex 4
C = 0.75
N = 500

plt.figure (figsize=(15,10))

G = nx.DiGraph ()
G.add_edge (0,0)
for i in range (1, N):
    in_degs = G.in_degree ()
    total = sum (list(dict(in_degs).values())) + len (G.nodes ()) * C
    rv = random.random () * total
    for node in G.nodes ():
        if rv < (in_degs[node] + C):
            G.add_edge (i, node)
            break
        rv -= (in_degs[node] + C)
nx.draw (G)
plt.show ()

plt.xscale ('log')

plt.hist (list (dict(G.in_degree()).values()), bins=np.logspace(0,5,10, base=2), log=True)
plt.show ()


In [ ]:
# ex. 2 (VI.1)

def spectralBi (G, n1):
    L = nx.laplacian_matrix (G).asfptype().todense ()
    nodes = list(G.nodes ().keys())
    n = len (nodes)
    
    lamb, vect = np.linalg.eig (L)
    ind = np.argmin (lamb)
    lamb[ind] = 100000
    ind = np.argmin (lamb)
    
    
    v2 = vect[:, ind]
    vals = []
    for i in range (len (v2)):
        vals.append ((v2[i], i))
    vals.sort ()
    
    
    p1 = {}
    p2 = {}
    for i in range (n1):
        p1[nodes[vals[i][1]]] = 1
        p2[nodes[vals[n - i - 1][1]]] = 1
    for i in range (n1, n):
        p1[nodes[vals[i][1]]] = 0
        p2[nodes[vals[n - i - 1][1]]] = 0
        
    if R (G, p1) < R (G, p2):
        return p1
    else:
        return p2
    
        
def R (G, p):
    r = 0
    for n1 in G.nodes ():
        for n2 in G.nodes ():
            if p[n1] != p[n2] and G.has_edge (n1, n2):
                r += 1 # for weighted graphs, should be incremented by weight
    return r
        

G = nx.karate_club_graph ()
res = spectralBi (G, 17)
pos = nx.spring_layout (G)
nx.draw_networkx_nodes (G, pos, [n for n in G.nodes() if res[n] == 1], node_color='r')
nx.draw_networkx_nodes (G, pos, [n for n in G.nodes() if res[n] != 1], node_color='b')
nx.draw_networkx_edges (G, pos)
nx.draw_networkx_labels (G, pos)
plt.show ()

In [ ]:
#!pip3 install python-louvain
import community
import matplotlib as mpl

def partition_and_draw (G):
    pos = nx.spring_layout(G)
    partition = community.best_partition(G)
    size = float(len(set(partition.values())))
    count = 0.
    cm = plt.cm.ScalarMappable(cmap = plt.get_cmap('Paired'), 
                                norm = mpl.colors.Normalize(vmin = 0, vmax = size, clip = False))
    print ("number of communities: ", int(size), " modularity: ", community.modularity (partition, G))
    for com in set(partition.values()) :
        count = count + 1.
        list_nodes = [nodes for nodes in partition.keys()
                                    if partition[nodes] == com]
        nx.draw_networkx_nodes(G, pos, list_nodes, node_size = 60, node_color = cm.to_rgba (count))


    nx.draw_networkx_edges(G,pos, alpha=0.5)
    plt.show()
    
partition_and_draw (nx.karate_club_graph())
partition_and_draw (nx.erdos_renyi_graph (33, 0.3))

In [ ]:
# ex VI.2

def modularity (G, p):
    q = 0.0
    degs = dict(nx.degree (G))
    m2 = 1.0 * sum (degs.values ())
    for n1 in G.nodes ():
        for n2 in G.nodes ():
            if p[n1] == p[n2]:
                if G.has_edge (n1, n2):
                    q += 1 # for weighted graphs, should be incremented by weight
                q -= degs[n1] * degs[n2] / m2
    q /= m2
    return q

print (modularity (G, res))

In [ ]:
# Ex.VII.1

G = nx.karate_club_graph ()
n = len (G.nodes())
x0 = np.random.rand (1, n)

L = nx.laplacian_matrix (G).todense ()

x = x0
dt = 0.0001
t = 0.0
while t < 100:
    x = x - dt * np.dot (x, L)
    t += dt

print (x)
print (np.sum (x0) / n)

In [ ]:
# Ex.VII.2

n = 300
G = nx.Graph ()
G.add_nodes_from (range (n))

dynamics_dic = {}
for i in range (n):
    dynamics_dic[i] = []
    for j in range (n):
        p = 0.02
        if i / 100 == j / 100:
            p = 0.8
        if random.random () < p:
            G.add_edge (i, j)

x0 = []
for i in range (n):
    x0.append (random.random () + int(i / 100) * 0.2 )
L = nx.laplacian_matrix (G).todense ()


x = np.array (x0)
dt = 0.001
t = 0.0
times = []
while t < 10:
    times.append (t)
    for i in range (n):
        dynamics_dic[i].append (x.item(i))
    x = x - dt * np.dot (x, L)
    t += dt   

plt.figure (figsize=(15,15))
for i in range (100):
    plt.plot (times, dynamics_dic[i], 'r')
for i in range (100, 200):
    plt.plot (times, dynamics_dic[i], 'g')
for i in range (200, 300):
    plt.plot (times, dynamics_dic[i], 'b')
plt.xscale ('log')
plt.show ()

In [ ]:
# ex. VIII.1


def pageRank (G, alpha):
    A = nx.adjacency_matrix (G)
    A = A.todense ()
    row_sums = np.sum (A, axis=1)
    T = A / (1.0 * row_sums)

    n = len (G.nodes())
    u = np.ones ((1, n)) / n

    p = np.ones ((1, n)) / n
    for i in range (1000):
        p = alpha * np.dot (p, T) + (1 - alpha) * u

    return p

plt.figure (figsize=(15,15))
G = nx.karate_club_graph ()
for alpha in np.arange (0, 1.00001, 0.1):
    p = pageRank (G, alpha)
    plt.plot ([p.item (ci) for ci in range (len (G.nodes()))], label=str (alpha))
plt.legend ()
plt.show ()


plt.figure (figsize=(15,15))
G = nx.random_regular_graph (5, 30)
p = pageRank (G, 0.85)
plt.plot ([p.item (ci) for ci in range (len (G.nodes()))])
plt.show ()